# DataProjectLab - Projet Sante
## Prevision de la demande en medicaments
### Guide Power BI - Dashboard decisionnel HGU Cocody
### VERSION CORRIGEE - Mode formateur

---

> **Comment lire ce guide :**
> Les blocs `POURQUOI` justifient chaque choix de visuel et d'architecture.
> Les blocs `CLINIQUE` expliquent ce que le Dr. Konan doit lire sur chaque page.
> Les blocs `ATTENTION` signalent les erreurs frequentes a eviter.

---

### Objectif du dashboard

Ce dashboard de 4 pages permet au Dr. Konan et a la direction de l'HGU Cocody de :
- Suivre les stocks en temps reel et detecter les ruptures imminentes
- Visualiser les tendances de consommation par medicament et par service
- Consulter les previsions a 4 semaines generees par le modele ML (NB3)
- Declencher les commandes au bon moment avec les bons fournisseurs

### Fichiers necessaires

| Fichier | Source | Usage |
|---|---|---|
| `medicaments.csv` | Dataset brut | Referentiel medicaments |
| `services.csv` | Dataset brut | Referentiel services |
| `fournisseurs.csv` | Dataset brut | Referentiel fournisseurs |
| `commandes_fournisseurs.csv` | Dataset brut | Historique commandes |
| `consommations.csv` | Dataset brut | Historique consommations |
| `previsions_4semaines.csv` | **Genere NB3** | Previsions ML + alertes stock |
| `stock_securite_optimise.csv` | **Genere NB3** | Parametres stock recalcules |

> **Ajouter `stock_securite_optimise.csv`** qui contient les ROP et EOQ calcules en NB3.
> Ce fichier manque dans le guide original mais est indispensable pour la Page 4.

---

# Etape 0 - Parametrage initial

## 0.1 - Chargement des fichiers

> **POURQUOI charger dans cet ordre ?**
>
> Power BI construit le modele de donnees incrementalement.
> En chargeant les tables de reference (medicaments, services, fournisseurs)
> avant les tables de faits (consommations, commandes), les relations
> sont plus faciles a configurer car les cles primaires sont deja presentes.
>
> **Accueil -> Obtenir des donnees -> Texte/CSV**

Charger dans cet ordre :
1. `medicaments.csv`
2. `services.csv`
3. `fournisseurs.csv`
4. `commandes_fournisseurs.csv`
5. `consommations.csv`
6. `rupture_stock.csv`
7. `previsions_4semaines.csv`
8. `stock_securite_optimise.csv`

> **CLINIQUE - Pourquoi `stock_securite_optimise.csv` est indispensable ?**
>
> Ce fichier contient les ROP (Reorder Point) calcules en NB3.
> Sans lui, le tableau d'alertes de la Page 4 ne peut pas comparer
> le stock actuel au seuil de commande recommande.
> C'est la difference entre un dashboard decoratif et un outil
> qui declenche des commandes au bon moment.

## 0.2 - Nettoyage dans Power Query

> **POURQUOI nettoyer les types dans Power Query et non apres ?**
>
> Un type mal defini au chargement se propage dans toutes les mesures DAX.
> Un `date_commande` charge comme texte empechera `PREVIOUSMONTH()` de fonctionner.
> Un `medicament_critique` en texte ('True'/'False') ne sera pas filtrable
> avec `= TRUE()` en DAX - il faudra `= "True"` ce qui est fragile.
>
> **Regle : corriger les types a la source (Power Query), jamais en DAX.**

| Table | Colonne | Type cible | Pourquoi |
|---|---|---|---|
| consommations | `date` | Date | Activer les analyses temporelles |
| consommations | `quantite_consommee` | Nombre entier | Eviter les decimales parasites |
| commandes_fournisseurs | `date_commande` | Date | Jointure avec Calendrier |
| commandes_fournisseurs | `date_livraison_reelle` | Date | Calcul des retards |
| commandes_fournisseurs | `retard_jours` | Nombre entier | Comparaisons DAX |
| medicaments | `prix_unitaire_euro` | Nombre decimal | Calculs financiers |
| medicaments | `medicament_critique` | Vrai/Faux | Filtres `= TRUE()` en DAX |
| medicaments | `stock_actuel` | Nombre entier | Comparaisons avec ROP |
| medicaments | `stock_securite` | Nombre entier | Calcul alertes |
| previsions_4semaines | `semaine` | Nombre entier | Filtres `= 1`, `= 2` |
| previsions_4semaines | `quantite_prevue` | Nombre decimal | Agregations |
| stock_securite_optimise | `point_commande_ROP` | Nombre entier | Comparaison stock |
| stock_securite_optimise | `qte_commande_EOQ` | Nombre entier | Calcul bon commande |

> **ATTENTION - Le statut de livraison a des accents.**
>
> Le champ `statut` dans `commandes_fournisseurs` contient 'Livrée'.
> Verifier la valeur exacte avec un filtre dans Power Query avant d'ecrire
> les mesures DAX. Un espace invisible ou un accent different
> fera que `= 'Livree'` ne retournera aucune ligne.

## 0.3 - Table Calendrier

> **POURQUOI la table Calendrier est obligatoire ?**
>
> Sans table Calendrier, les fonctions DAX `PREVIOUSMONTH()`, `DATESYTD()`,
> `SAMEPERIODLASTYEAR()` ne fonctionnent pas.
> Ces fonctions ont besoin d'une table de dates continue sans trou.
> `consommations[date]` ne peut pas jouer ce role car tous les jours
> ne sont pas representes (weekends, jours sans consommation enregistree).
>
> **Dans Modelisation -> Nouvelle table, coller ce code DAX :**

```dax
Calendrier = 
ADDCOLUMNS(
    CALENDAR(DATE(2022,1,1), DATE(2024,9,30)),
    "Annee",        YEAR([Date]),
    "Mois_Num",     MONTH([Date]),
    "Mois_Nom",     FORMAT([Date], "MMM", "fr-FR"),
    "Trimestre",    "T" & QUARTER([Date]),
    "Semaine",      WEEKNUM([Date]),
    "Jour_Semaine", FORMAT([Date], "dddd", "fr-FR"),
    "Est_Weekend",  IF(WEEKDAY([Date],2)>=6, TRUE(), FALSE()),
    "Annee_Mois",   FORMAT([Date], "YYYY-MM")
)
```

> **ATTENTION - Marquer la table comme table de dates.**
>
> Apres creation : clic droit sur `Calendrier` -> Marquer en tant que table de dates -> Date.
> Sans cette etape, `PREVIOUSMONTH()` et les autres fonctions temporelles
> ne fonctionneront pas. C'est l'etape la plus souvent oubliee.
>
> **Etendre jusqu'a DATE(2024,9,30) et non DATE(2024,6,30) :**
> Les previsions generees en NB3 couvrent juillet-aout 2024.
> La table Calendrier doit couvrir toute la periode des previsions.

## 0.4 - Modele de donnees (schema en etoile)

> **POURQUOI le schema en etoile est optimal pour Power BI ?**
>
> Power BI (moteur VertiPaq) est optimise pour ce pattern.
> Les jointures sont pre-calculees, les filtres se propagent
> automatiquement de la dimension vers les faits.
> Un filtre sur `fournisseurs[nom]` filtre automatiquement
> toutes les commandes de ce fournisseur dans les visuels.
>
> **Dans Vue Modele, creer ces 7 relations (toutes 1-a-plusieurs) :**

```
Calendrier[Date]              -> consommations[date]           (1:N)
Calendrier[Date]              -> commandes_fournisseurs[date_commande] (1:N)
medicaments[id_medicament]    -> consommations[id_medicament]  (1:N)
medicaments[id_medicament]    -> commandes_fournisseurs[id_medicament] (1:N)
medicaments[id_medicament]    -> previsions_4semaines[id_medicament]   (1:N)
medicaments[nom]              -> stock_securite_optimise[medicament]   (1:1)
services[id_service]          -> consommations[id_service]    (1:N)
fournisseurs[id_fournisseur]  -> commandes_fournisseurs[id_fournisseur] (1:N)
```

> **Relation ajoutee : `medicaments[nom]` -> `stock_securite_optimise[medicament]`**
>
> Cette relation permet aux mesures DAX de la Page 4 d'acceder
> au ROP et a l'EOQ calcules en NB3 pour chaque medicament.
>
> **Direction des filtres : toujours Single (pas Both).**
> La direction Both cree des cycles de filtres et des resultats inattendus.
> Garder Single sauf besoin explicitement justifie.

---

# Etape 1 - Mesures DAX

> **POURQUOI creer les mesures avant les visuels ?**
>
> En Power BI, on peut glisser une colonne directement dans un visuel.
> Mais une mesure DAX est plus puissante : elle s'adapte au contexte de filtre,
> peut combiner plusieurs tables et applique des regles metier.
>
> `SUM(consommations[quantite_consommee])` direct dans un visuel ne peut pas
> etre reutilise, conditionne ou documente.
> La meme chose comme mesure `[Total Unites Consommees]` peut etre
> reutilisee dans 10 visuels, combinee avec `CALCULATE`, conditionnee.
>
> **Creer une table vide `_Mesures` :**
> Modelisation -> Nouvelle table -> `_Mesures = ROW("vide",0)` puis supprimer la colonne.
> L'underscore place la table en premier dans le panneau des champs.
> Toutes les mesures y seront rangees.

## Mesures de base

> **POURQUOI `SUMX(ruptures_stock, ...)` et non `SUM(ruptures_stock[duree_jours])` ?**
>
> `SUM` additionne une seule colonne.
> `SUMX` itere sur chaque ligne et applique une expression.
> Pour `Cout Ruptures Estime = patients x jours x 50 euros`, il faut SUMX
> car le calcul est une multiplication de 3 colonnes par ligne.
>
> **`AVERAGEX(FILTER(...), ...)` pour le retard moyen :**
> On filtre d'abord sur `retard_jours > 0` pour ne calculer la moyenne
> que sur les commandes effectivement retardees.
> Sans ce filtre, les 0 (commandes a temps) dilueraient la moyenne
> et sous-estimeraient le vrai impact des retards.

```dax
-- Mesure 1 : Total des unites consommees (toutes tables)
Total Unites Consommees = 
SUM(consommations[quantite_consommee])

-- Mesure 2 : Cout total de consommation
Cout Total Consommation = 
SUM(consommations[cout_euro])

-- Mesure 3 : Jours de rupture (hors impact mineur)
Nb Jours Rupture = 
CALCULATE(
    SUMX(ruptures_stock, ruptures_stock[duree_jours]),
    ruptures_stock[impact_clinique] <> "Impact mineur"
)

-- Mesure 4 : Patients affectes par des ruptures
Patients Affectes Total = 
SUM(ruptures_stock[patients_affectes])

-- Mesure 5 : Taux de livraison a temps (sur commandes livrees)
-- DIVIDE securise la division si denominateur = 0
Taux Livraison a Temps = 
VAR livrees = 
    CALCULATE(
        COUNTROWS(commandes_fournisseurs),
        commandes_fournisseurs[statut] = "Livrée"
    )
VAR a_temps = 
    CALCULATE(
        COUNTROWS(commandes_fournisseurs),
        commandes_fournisseurs[statut] = "Livrée",
        commandes_fournisseurs[retard_jours] = 0
    )
RETURN
    DIVIDE(a_temps, livrees)

-- Mesure 6 : Retard moyen sur les commandes retardees uniquement
-- AVERAGEX + FILTER = moyenne conditionnelle (ignorant les 0)
Retard Moyen Jours = 
AVERAGEX(
    FILTER(commandes_fournisseurs, commandes_fournisseurs[retard_jours] > 0),
    commandes_fournisseurs[retard_jours]
)
```

> **CLINIQUE - Ces 6 mesures repondent directement au brief du Dr. Konan :**
>
> `Cout Total Consommation` : 'Combien ca nous coute ?' (~4M euros sur 30 mois)
> `Nb Jours Rupture` : 'Combien de jours en rupture ?' (KPI a faire descendre)
> `Patients Affectes Total` : 'Combien de patients touches ?' (KPI clinique prioritaire)
> `Taux Livraison a Temps` : 'Nos fournisseurs sont-ils fiables ?' (38-40% = alerte)
> `Retard Moyen Jours` : 'Combien de jours de retard en moyenne ?' (2 jours)

## Mesures avancees

> **POURQUOI `PREVIOUSMONTH()` necessite la table Calendrier marquee ?**
>
> `PREVIOUSMONTH(Calendrier[Date])` retourne les dates du mois precedent
> par rapport au contexte de filtre actuel (le mois selectionne dans un slicer).
> Sans la table Calendrier marquee comme table de dates, Power BI ne sait pas
> quelle dimension temporelle utiliser et la mesure retourne un resultat vide.
>
> **`VAR ... RETURN` dans `Statut Stock` :**
> Les variables DAX (`VAR`) calculent une valeur une seule fois et la reutilisent.
> Sans VAR, `stock_actuel` serait calcule 4 fois dans les 4 SWITCH.
> Avec VAR : calcule une fois, reutilise 4 fois = plus lisible et plus performant.

```dax
-- Mesure 7 : Consommation du mois precedent (pour variation)
Consommation Mois Precedent = 
CALCULATE(
    [Total Unites Consommees],
    PREVIOUSMONTH(Calendrier[Date])
)

-- Mesure 8 : Variation mensuelle en %
Variation Mensuelle Pct = 
DIVIDE(
    [Total Unites Consommees] - [Consommation Mois Precedent],
    [Consommation Mois Precedent]
)

-- Mesure 9 : Cout estime des ruptures (patients x jours x 50 euros)
-- Hypothese : 50 euros de cout indirect par patient par jour de rupture
Cout Ruptures Estime = 
SUMX(
    ruptures_stock,
    ruptures_stock[patients_affectes] * ruptures_stock[duree_jours] * 50
)

-- Mesure 10 : Prevision semaine 1 (pour les alertes de commande)
Prevision Semaine 1 = 
CALCULATE(
    SUM(previsions_4semaines[quantite_prevue_semaine]),
    previsions_4semaines[semaine] = 1
)

-- Mesure 11 : Prevision semaine 2 (pour anticiper la commande suivante)
Prevision Semaine 2 = 
CALCULATE(
    SUM(previsions_4semaines[quantite_prevue_semaine]),
    previsions_4semaines[semaine] = 2
)

-- Mesure 12 (AJOUTEE) : ROP depuis stock_securite_optimise
-- Ramene le point de commande calcule en NB3 pour la comparaison
ROP Recommande = 
MAX(stock_securite_optimise[point_commande_ROP])

-- Mesure 13 (AJOUTEE) : Quantite a commander selon EOQ
EOQ Recommande = 
MAX(stock_securite_optimise[qte_commande_EOQ])

-- Mesure 14 : Statut Stock (mesure cle pour la Page 4)
-- VAR calcule chaque valeur une seule fois et la reutilise dans SWITCH
Statut Stock = 
VAR stock_actuel = MAX(medicaments[stock_actuel])
VAR stock_secu   = MAX(medicaments[stock_securite])
VAR rop          = [ROP Recommande]
VAR prev_s1      = [Prevision Semaine 1]
RETURN
    SWITCH(
        TRUE(),
        stock_actuel < stock_secu,         "RUPTURE IMMINENTE",
        stock_actuel < rop,                "COMMANDER CETTE SEMAINE",
        stock_actuel < rop + prev_s1,      "SURVEILLER",
        "Stock OK"
    )
```

> **ATTENTION - Utiliser le ROP calcule en NB3, pas le stock_securite brut.**
>
> La mesure `Statut Stock` doit utiliser le `point_commande_ROP` calcule
> par la formule `conso_moy x delai + stock_securite_calcule`.
> Sinon le tableau d'alertes continuera a afficher 'Stock OK' pour
> l'Insuline (stock=8, ROP=124) - ce qui est faux et dangereux.

---

# Page 1 - Vue d'ensemble executive

**Titre :** 'Tableau de bord Pharmacie - HGU Cocody'

> **CLINIQUE - Ce que le Dr. Konan doit voir en 30 secondes sur cette page :**
>
> 1. Est-ce que le budget medicaments est dans les clous ?
> 2. Combien de ruptures ce mois ?
> 3. Combien de patients ont ete affectes ?
> 4. Les fournisseurs livrent-ils a temps ?
>
> **Si la reponse a l'une de ces 4 questions est alarmante,
> le Dr. Konan doit le voir sans avoir a chercher.**

### Disposition recommandee

```
+----------------------------------------------------------+
| LOGO HGU | Titre | Filtres : Annee / Mois / Medicament  |
+----------+--------+----------+---------------------------+
| KPI 1    | KPI 2  | KPI 3    | KPI 4                    |
| Cout tot | Ruptur | Patients | Tx livraison a temps     |
+----------+--------+----------+---------------------------+
|                                                          |
|    Graphique lineaire - Evolution mensuelle conso        |
|    + ligne de tendance + variation MoM                   |
|                                                          |
+---------------------------+------------------------------+
| Barres horizontales       | Anneau                       |
| Top 8 medicaments (cout)  | Repartition categories       |
+---------------------------+------------------------------+
```

### Construction Page 1

**1. Les 4 cartes KPI**

> **POURQUOI les KPIs en haut ?**
>
> L'oeil scanne de haut en bas. Les KPIs sont les chiffres les plus importants
> - ils doivent etre vus en premier, en 2 secondes.
> Un dashboard qui commence par un graphique force l'utilisateur
> a chercher les chiffres cles.

- **KPI 1 - Cout Total Consommation**
  - Mesure : `[Cout Total Consommation]` · Format : `0.0 M€`
  - Couleur normale : vert (#1D9E75)
  - Etiquette : 'Cout total consommation'

- **KPI 2 - Nb Jours Rupture**
  - Mesure : `[Nb Jours Rupture]`
  - Couleur conditionnelle : rouge (#E24B4A) si > 30, vert (#1D9E75) si <= 10
  - Etiquette : 'Jours de rupture (hors impact mineur)'
  - Cible a afficher : 0 (objectif zero rupture critique)

- **KPI 3 - Patients Affectes**
  - Mesure : `[Patients Affectes Total]`
  - Couleur : orange (#BA7517) si > 0
  - Etiquette : 'Patients affectes par des ruptures'

- **KPI 4 - Taux Livraison a Temps**
  - Mesure : `[Taux Livraison a Temps]` · Format : Pourcentage
  - Couleur : rouge si < 50%, orange si < 80%, vert si >= 90%
  - Cible a afficher : 90%


**2. Graphique lineaire - Evolution mensuelle**

> **POURQUOI deux courbes (actuel + mois precedent) ?**
>
> Une seule courbe montre le niveau absolu mais pas la direction.
> Ajouter `[Consommation Mois Precedent]` en pointilles permet de voir
> immediatement si ce mois est au-dessus ou en dessous du mois precedent.
> C'est l'information operationnelle que le Dr. Konan regarde chaque lundi.

- Type : Graphique en courbes
- Axe X : `Calendrier[Annee_Mois]` (et non `consommations[date]` brut)
- Valeur Y principale : `[Total Unites Consommees]` · Couleur : vert (#1D9E75)
- Valeur Y secondaire : `[Consommation Mois Precedent]` · Style : pointilles · Gris (#888780)
- Activer 'Ligne de tendance' dans les options analytiques
- Etiquette de donnee sur le dernier point uniquement

> **ATTENTION - Utiliser `Calendrier[Annee_Mois]` comme axe X.**
>
> Si on utilise `consommations[date]` directement, Power BI affiche
> un point par jour (trop granulaire). `Calendrier[Annee_Mois]`
> regroupe automatiquement par mois grace a la relation.

**3. Barres horizontales - Top 8 medicaments**

- Type : Graphique a barres groupees (horizontal)
- Axe Y : `medicaments[nom]` · Filtre Top N = 8 par `[Cout Total Consommation]`
- Valeur X : `[Cout Total Consommation]`
- Couleur conditionnelle sur `medicaments[medicament_critique]` :
  rouge (#E24B4A) si TRUE, bleu (#185FA5) si FALSE
- Trier par valeur decroissante (barre la plus longue en haut)

> **CLINIQUE - Ce graphique montre que les 3 medicaments les plus couteux
> (Insuline 21.6%, Heparine 20.4%, Erythropoietine 15.3%) sont tous en rouge.
> 57.3% du budget est concentre sur 3 references critiques.**

**4. Anneau - Repartition par categorie**

- Type : Graphique en anneau
- Legende : `medicaments[categorie]`
- Valeurs : `[Cout Total Consommation]`
- Cercle central : afficher le total en euros

---

# Page 2 - Analyse des consommations par service

**Titre :** 'Consommation par service hospitalier'

> **CLINIQUE - Ce que le Dr. Konan doit voir sur cette page :**
>
> Quel service consomme le plus ? Sur quel medicament ?
> La Reanimation (30.4% du budget) et l'Oncologie (19.5%)
> doivent apparaitre clairement comme les services prioritaires.
> Un pic inhabituel dans un service = signal d'une epidemie
> ou d'un changement de protocole a investiguer.

### Construction Page 2

**1. Matrice heatmap - Services x Medicaments**

> **POURQUOI une matrice et non un graphique a barres ?**
>
> La matrice permet de voir simultanement 10 services x 20 medicaments
> = 200 combinaisons en une seule vue. Un graphique a barres ne pourrait
> afficher que l'une de ces dimensions a la fois.
> Le formatage conditionnel transforme la matrice en heatmap visuelle.

- Type : **Matrice**
- Lignes : `services[nom_service]`
- Colonnes : `medicaments[nom]` - filtrer sur `medicament_critique = TRUE`
- Valeurs : `[Total Unites Consommees]`
- Formatage conditionnel : Mise en forme -> Cellules -> Arriere-plan
  - Echelle : blanc (min) -> bleu (#185FA5) (max)
  - Activer pour les valeurs et les en-tetes

> **ATTENTION - Filtrer sur les medicaments critiques uniquement.**
>
> Avec 22 medicaments, la matrice devient illisible.
> Filtrer sur `medicament_critique = TRUE` reduit a ~20 colonnes.
> Ou utiliser un slicer sur la page pour que le Dr. Konan
> puisse selectionner les medicaments qui l'interessent.

**2. Barres empilees - Evolution mensuelle par service**

- Type : Graphique a barres empilees
- Axe X : `Calendrier[Annee_Mois]`
- Legende : `services[nom_service]`
- Valeurs : `[Total Unites Consommees]`
- Palette : une couleur distincte par service (10 services = 10 couleurs)

> **CLINIQUE - Ce graphique permet de detecter :**
>
> - Un service qui grossit rapidement (surface qui augmente dans les barres)
> - Un evenement exceptionnel (pic inhabituel dans un service un mois)
> - La tendance de +0.85%/mois globale (+1.8%/mois pour l'Insuline)

**3. Treemap - Poids relatif des services**

- Type : Treemap
- Groupe : `services[departement]`
- Details : `services[nom_service]`
- Valeurs : `[Cout Total Consommation]`

> **Lecture attendue :** Reanimation (30.4%) = rectangle le plus grand.
> Oncologie (19.5%) = deuxieme. Neurologie (1.7%) = le plus petit.

**4. Tableau detaille filtrable**

- Type : Table
- Colonnes : Medicament · Service · Mois · Unites · Cout · `[Variation Mensuelle Pct]`
- Tri par defaut : Cout decroissant
- Formatage conditionnel sur Variation : fleche verte si > 0, rouge si < 0

---

# Page 3 - Fournisseurs & Commandes

**Titre :** 'Performance fournisseurs & gestion des commandes'

> **CLINIQUE - Ce que le Dr. Konan doit voir sur cette page :**
>
> Aucun fournisseur n'atteint la cible de 90% de ponctualite (38-40% reel).
> Cette page transforme ce constat en decisions concretes :
> quel fournisseur privilegier ? Pour quel medicament ? Avec quel delai ?

### Construction Page 3

**1. Jauge - Taux de livraison a temps**

> **POURQUOI une jauge et non une carte ?**
>
> La jauge materialise visuellement la distance a l'objectif (90%).
> Voir une aiguille a 39% sur une jauge dont la zone verte commence a 90%
> est plus impactant qu'un chiffre '39%' sur une carte.
> Le Dr. Konan comprend immediatement que quelque chose ne va pas.

- Type : **Jauge**
- Valeur : `[Taux Livraison a Temps]`
- Valeur cible : 0.90
- Min : 0 · Max : 1
- Zone rouge : 0 a 0.50 · Zone orange : 0.50 a 0.80 · Zone verte : 0.80 a 1.0

> **ATTENTION - La jauge affichera du rouge profond (38-40%).**
>
> C'est la realite des donnees NB2. Ne pas ajuster la cible a la baisse
> pour 'faire mieux paraître' le dashboard. La mission du dashboard
> est de montrer la realite, meme inconfortable.

**2. Barres - Retard moyen par fournisseur**

- Type : Graphique a barres groupees
- Axe Y : `fournisseurs[nom]`
- Valeur X : `[Retard Moyen Jours]`
- Couleur conditionnelle : rouge si > 3 jours, orange si > 1 jour
- Annotation : afficher la valeur en bout de barre

> **Resultat attendu :** toutes les barres autour de 2 jours.
> EuroPharma avec 21 jours de delai contractuel = meme retard que
> PharmaDistrib avec 4 jours. Le delai ne predit pas la ponctualite.

**3. Nuage de points - Fiabilite vs Delai**

- Type : Graphique a nuages de points
- Axe X : `fournisseurs[delai_livraison_jours]`
- Axe Y : `[Taux Livraison a Temps]` par fournisseur
- Taille : `SUM(commandes_fournisseurs[montant_total_euro])`
- Etiquettes : `fournisseurs[nom]`
- Ligne de reference horizontale a 90% (cible)
- `set_ylim` : forcer l'axe Y de 0 a 100%

> **ATTENTION - Forcer l'axe Y de 0 a 100%.**
>
> Si les 5 fournisseurs ont tous un taux entre 37% et 40%,
> Power BI va zoomer automatiquement sur cette plage (36%-41%).
> Les points seront disperses visuellement mais tres proches en realite.
> Forcer 0-100% montre la vraie situation : tous tres loin de la cible.

**4. Timeline des commandes**

- Type : Graphique en courbes
- Axe X : `Calendrier[Annee_Mois]`
- Valeurs : `SUM(commandes_fournisseurs[montant_total_euro])`
- Filtre : `statut = 'Livree'` uniquement
- Avec slicer sur `fournisseurs[nom]` pour filtrer par fournisseur

---

# Page 4 - Previsions & Alertes de stock

**Titre :** 'Previsions 4 semaines - Alertes de reapprovisionnement'

> **CLINIQUE - C'est la page la plus importante operationnellement.**
>
> Le Dr. Konan l'ouvre chaque lundi matin. En 2 minutes, il doit savoir :
> - Quels medicaments doivent etre commandes cette semaine ?
> - En quelle quantite ?
> - Aupres de quel fournisseur ?
>
> Cette page transforme les previsions ML du NB3 en decisions d'achat.

### Construction Page 4

**1. Tableau d'alerte - Medicaments a commander**

> **POURQUOI le tableau d'alerte est le visuel central de cette page ?**
>
> Tous les autres visuels (courbes, barres) sont des analyses.
> Le tableau d'alerte est une **decision**.
> Il dit au Dr. Konan exactement ce qu'il doit faire sans
> qu'il ait besoin d'interpreter un graphique.

- Type : **Table**
- Colonnes :
  - `medicaments[nom]` (Medicament)
  - `medicaments[stock_actuel]` (Stock actuel)
  - `[ROP Recommande]` (Point de commande NB3)
  - `[Prevision Semaine 1]` (Besoin S+1)
  - `[Prevision Semaine 2]` (Besoin S+2)
  - `[EOQ Recommande]` (Quantite a commander)
  - `[Statut Stock]` (Alerte coloree)
- Tri : `[Statut Stock]` en premier (RUPTURE en haut)
- Formatage conditionnel sur `[Statut Stock]` :
  - 'RUPTURE IMMINENTE' : fond rouge (#E24B4A) texte blanc
  - 'COMMANDER CETTE SEMAINE' : fond orange (#BA7517) texte blanc
  - 'SURVEILLER' : fond jaune (#F5D76E) texte noir
  - 'Stock OK' : fond vert (#1D9E75) texte blanc

> **CLINIQUE - Lecture attendue du tableau :**
>
> Insuline (stock=8, ROP=124) -> RUPTURE IMMINENTE en rouge.
> Heparine (stock=7, ROP=54) -> RUPTURE IMMINENTE en rouge.
> Salbutamol (stock=10, ROP=239) -> RUPTURE IMMINENTE en rouge.
> Ces 3 medicaments doivent declencher une commande aujourd'hui.

**2. Graphique lineaire - Historique + Previsions**

> **POURQUOI combiner historique et previsions sur le meme graphique ?**
>
> L'historique donne le contexte (ou on etait).
> Les previsions donnent la direction (ou on va).
> Les bornes de confiance donnent l'incertitude (combien de marge prendre).
> Sans ces 3 elements ensemble, la prevision seule est peu credible.

- Type : Graphique en courbes
- Axe X : Dates (historique + previsions)
- Serie 1 : Historique (`[Total Unites Consommees]`) · Bleu (#185FA5) · Trait plein
- Serie 2 : Prevision (`SUM(previsions_4semaines[quantite_prevue])`) · Orange (#BA7517) · Pointilles
- Serie 3 : Borne basse (`SUM(previsions_4semaines[borne_basse])`) · Rouge transparent · Remplissage
- Serie 4 : Borne haute (`SUM(previsions_4semaines[borne_haute])`) · Rouge transparent · Remplissage
- Ligne verticale a la date d'aujourd'hui (separation passe/futur)

**3. Barres - Quantites a commander par fournisseur**

- Type : Graphique a barres groupees
- Axe Y : `fournisseurs[nom]`
- Valeur X : Somme des `[EOQ Recommande]` pour les medicaments
  avec `[Statut Stock]` = 'RUPTURE IMMINENTE' ou 'COMMANDER CETTE SEMAINE'
- Annotation : delai de livraison sur chaque barre

> **CLINIQUE - Ce graphique repond a la question operationnelle :**
>
> 'A qui passer commande et combien ?' en un seul coup d'oeil.
> EuroPharma apparaitra avec le volume le plus important (21j delai)
> car il fournit l'Insuline et l'Erythropoietine - les plus couteuses.

**4. KPIs globaux de la page**

- **Nb medicaments en alerte rouge** :
  `Nb medicaments en alerte rouge = 
CALCULATE(
    COUNTROWS(medicaments),
    FILTER(
        medicaments,
        [Statut Stock] = "RUPTURE IMMINENTE"
    )
)`
- **Valeur totale a commander (euros)** :
  `SUMX(FILTER(medicaments, [Statut Stock] IN {"RUPTURE IMMINENTE","COMMANDER CETTE SEMAINE"}),
  [EOQ Recommande] * MAX(medicaments[prix_unitaire_euro]))`
- **Fournisseur le plus urgent** : celui avec le plus de medicaments en alerte

---

# Charte graphique

> **POURQUOI une charte graphique stricte pour un dashboard hospitalier ?**
>
> La coherence visuelle reduit la charge cognitive : quand le Dr. Konan
> voit du rouge, il sait immediatement que c'est une alerte
> sans avoir a lire la legende. Cette automaticite est critique
> dans un contexte ou les decisions sont urgentes.
>
> Les memes codes couleur sont utilises dans les notebooks Python (COLORS)
> et dans le dashboard Power BI - coherence totale du projet.

### Couleurs

| Usage | Couleur | Code HEX |
|---|---|---|
| Couleur principale | Vert Ciel | #1D9E75 |
| Alerte critique - Rupture | Rouge | #E24B4A |
| Attention - Commander | Orange | #BA7517 |
| Surveiller | Jaune | #F5D76E |
| Bon etat - Stock OK | Vert | #1D9E75 |
| Neutre - Labels | Gris | #888780 |
| Fond des pages | Blanc casse | #F8F8F6 |
| Fond des cartes | Blanc | #FFFFFF |

> **Les 4 niveaux d'alerte ont une semantique fixe :**
> Rouge = agir aujourd'hui. Orange = agir cette semaine.
> Jaune = surveiller. Vert = aucune action necessaire.
> Cette semantique doit etre expliquee au Dr. Konan lors de la
> premiere presentation pour qu'elle devienne automatique.

### Typographie

- Titres de page : Segoe UI Semibold · 18pt · Couleur #2C2C2A
- Titres de visuels : Segoe UI · 12pt · Couleur #5F5E5A
- Valeurs KPI : Segoe UI Bold · 28pt

### Mise en forme des visuels

- Bordure arrondie (rayon 4px) sur tous les visuels
- Ombre legere activee (profondeur 4)
- Espacement entre visuels : 8px minimum
- Fond des pages : #F8F8F6 (blanc casse) - moins fatiguant que le blanc pur
  pour une lecture quotidienne

---

# Checklist finale

## Chargement et modele

- [ ] 7 fichiers CSV charges (dont `stock_securite_optimise.csv`)
- [ ] Types corriges dans Power Query pour toutes les tables
- [ ] Table Calendrier creee jusqu'a DATE(2024,9,30)
- [ ] Table Calendrier **marquee comme table de dates** (etape souvent oubliee)
- [ ] 8 relations definies dont `medicaments[nom] -> stock_securite_optimise[medicament]`
- [ ] Schema en etoile valide (pas de relation cyclique)

## Mesures DAX

- [ ] 14 mesures creees dans la table `_Mesures`
- [ ] `[Taux Livraison a Temps]` teste : doit retourner ~38-40%
- [ ] `[Cout Ruptures Estime]` teste : doit retourner ~250 000 euros
- [ ] `[Statut Stock]` teste sur Insuline : doit retourner 'RUPTURE IMMINENTE'
- [ ] `[ROP Recommande]` teste sur Insuline : doit retourner 124

## Pages

- [ ] Page 1 : 4 KPIs + graphique lineaire + barres + anneau
- [ ] Page 2 : Matrice heatmap + barres empilees + treemap + tableau
- [ ] Page 3 : Jauge (rouge a 38-40%) + retards + scatter + timeline
- [ ] Page 4 : Tableau alertes (3 rouges visibles) + courbe historique+previsions
- [ ] Charte graphique appliquee uniformement
- [ ] Filtres globaux : Annee et Medicament sur toutes les pages

## Validation finale avec le Dr. Konan

- [ ] Page 4 : Insuline, Heparine, Salbutamol en RUPTURE IMMINENTE (rouge)
- [ ] KPI 4 (Taux livraison) en rouge (~38-40%, cible 90%)
- [ ] Cout total ~4M euros sur 30 mois
- [ ] Previsions juillet 2024 en baisse vs juin (creux saisonnier attendu)

---

## Presentation de 5 minutes au Dr. Konan

> **Structure recommandee :**
>
> **Minute 1 - La situation actuelle (Page 1) :**
> 'Voici votre budget medicaments sur 30 mois : X euros.
> X jours de rupture ont affecte Y patients. Vos fournisseurs
> livrent a temps seulement 38-40% du temps.'
>
> **Minute 2 - Ou ca se passe (Page 2) :**
> 'La Reanimation et l'Oncologie concentrent 50% du budget.
> La heatmap montre les pics saisonniers par medicament.'
>
> **Minute 3 - Pourquoi les ruptures (Page 3) :**
> 'Aucun fournisseur n'atteint la cible de 90% de ponctualite.
> EuroPharma a 21 jours de delai - les commandes doivent partir
> 25 jours avant le besoin.'
>
> **Minute 4 - Ce qu'on fait (Page 4) :**
> 'Ces 3 medicaments necessitent une commande aujourd'hui.
> Les previsions a 4 semaines sont generees automatiquement chaque lundi.'
>
> **Minute 5 - Les 3 recommandations :**
> 1. Commander immediatement Insuline, Heparine, Salbutamol
> 2. Reviser les stocks de securite pour 9 medicaments
> 3. Utiliser ce dashboard chaque lundi matin pour les commandes

---

*DataProjectLab - apprendre la data sur des cas concrets, structures et orientes metier.*